# NYPD Complaint Data — Complete Analysis & Prediction (v2)**Dataset:** 2020–2025 NYPD Complaint Data (two CSV files merged)## Pipeline1. Load & Merge both CSVs (2020–2024 historical + 2025 YTD)2. Data Cleaning3. Exploratory Data Analysis — including year-over-year trends4. Feature Engineering5. Prediction: **LightGBM** + **CatBoost** across **5 target variables**6. Grand Comparison (10 model runs)## Why LightGBM + CatBoost?| Model | Why better than RF / XGBoost ||-------|------------------------------|| **LightGBM** | Leaf-wise growth + histogram binning → faster & more accurate on 2.5M rows || **CatBoost** | Native ordered target statistics → no label-encoding artifacts; best on mixed tabular data |## 5 Target Variables| # | Target | Task | Real-world value ||---|--------|------|------------------|| 1 | `LAW_CAT_CD` | 3-class (Felony / Misdemeanor / Violation) | Predict **severity** → triage resources || 2 | `CRM_ATPT_CPTD_CD` | Binary (Completed / Attempted) | Predict **outcome** → prevention || 3 | `BORO_NM` | 5-class | Predict **where** → geographic allocation || 4 | `OFNS_DESC` (top 10) | 10-class | Predict **crime type** → specialised unit routing || 5 | `DAILY_CASE_COUNT` | Regression | Predict **how many** per day → staffing |## v2 Key Fixes (vs Version 1)- **Dual CSV loading**: 2020–2024 + 2025 merged into one dataset (~2.5M rows)- **Leakage fix**: `OFNS_DESC` and `PD_DESC` removed from `LAW_CAT_CD` features (v1 had 99.97% accuracy — it was cheating)- **Imbalance fix**: `class_weight='balanced'` for targets with minority classes (v1 ATTEMPTED F1 was 0.02)- **Regression fix**: 5 years of daily data → LAG_365 now works; R² goes from −3.54 to meaningful- **YEAR added as feature**: captures multi-year trend shifts

---## 1. Setup & Imports

In [ ]:
!pip install catboost lightgbm --quiet

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import LabelEncoderfrom sklearn.metrics import (    classification_report, confusion_matrix, ConfusionMatrixDisplay,    accuracy_score, f1_score, precision_score, recall_score,    mean_absolute_error, mean_squared_error, r2_score)import lightgbm as lgbfrom catboost import CatBoostClassifier, CatBoostRegressorsns.set_theme(style='whitegrid', font_scale=1.05)pd.set_option('display.max_columns', 40)print('All libraries loaded.')

---## 2. Load & Merge DatasetsTwo CSV files:- `2020-2024_complaint_data.csv` — 5 years of historical data- `2025_complaint_data.csv` — current year (filtered to 2025 only)Both have the same core columns (different ordering). `pd.concat` aligns by column name automatically.

In [ ]:
# ── File paths ── update if running on a different machinePATH_HIST = '/Users/rahulraj1406/DADM/2020-2024_complaint_data.csv'PATH_2025 = '/Users/rahulraj1406/DADM/2025_complaint_data.csv'# ── Load 2020-2024 historical data ──print('Loading 2020-2024 historical data...')df_hist = pd.read_csv(PATH_HIST, low_memory=False)print(f'  Shape: {df_hist.shape}')# ── Load and filter 2025 data ──# The 2025 file may contain stray records from late 2024; filter strictly to 2025print('Loading 2025 data...')df_2025_raw = pd.read_csv(PATH_2025, low_memory=False)df_2025_raw['_yr'] = pd.to_datetime(df_2025_raw['CMPLNT_FR_DT'], errors='coerce').dt.yeardf_2025 = df_2025_raw[df_2025_raw['_yr'] == 2025].drop(columns=['_yr'])# Drop the extra column that only exists in the 2025 filedf_2025 = df_2025.drop(columns=['New Georeferenced Column'], errors='ignore')print(f'  2025 after year filter: {df_2025.shape}')# ── Concatenate — pd.concat aligns columns by name automatically ──df_raw = pd.concat([df_hist, df_2025], ignore_index=True, sort=False)print(f'\nCombined: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

---## 3. Understand the Dataset

In [ ]:
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')print(f'\nColumn names:\n{df_raw.columns.tolist()}')

In [ ]:
df_raw.head()

In [ ]:
df_raw.tail()

In [ ]:
df_raw.sample(5, random_state=42)

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
# ── Missing values: count + percentage ──missing = df_raw.isnull().sum().sort_values(ascending=False)missing_pct = (missing / len(df_raw) * 100).round(2)miss_df = pd.DataFrame({'Count': missing, 'Pct (%)': missing_pct})print(miss_df[miss_df['Count'] > 0].to_string())

In [ ]:
df_raw.nunique().sort_values(ascending=False)

---## 4. Data Cleaning

In [ ]:
df = df_raw.copy()print(f'Starting shape: {df.shape}')

In [ ]:
# ── 4.1  Parse dates & times ──df['CMPLNT_FR_DT'] = pd.to_datetime(df['CMPLNT_FR_DT'], errors='coerce')df['CMPLNT_TO_DT'] = pd.to_datetime(df['CMPLNT_TO_DT'], errors='coerce')df['CMPLNT_FR_TM'] = pd.to_datetime(df['CMPLNT_FR_TM'], format='%H:%M:%S', errors='coerce').dt.timeprint('Dates and times parsed.')

In [ ]:
# ── 4.2  Remove duplicates ──before = len(df)df = df.drop_duplicates()print(f'Duplicates removed: {before - len(df):,}')

In [ ]:
# ── 4.3  Remove rows where end date is before start date ──mask = (df['CMPLNT_TO_DT'] - df['CMPLNT_FR_DT']).dt.days < 0df = df[~mask]print(f'Inconsistent date rows removed: {mask.sum():,}')

In [ ]:
# ── 4.4  Remove invalid GPS coordinates (NYC bounding box) ──df['Latitude']  = pd.to_numeric(df['Latitude'],  errors='coerce')df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')before = len(df)df = df[df['Latitude'].between(40, 42) & df['Longitude'].between(-75, -72)]print(f'Invalid coordinates removed: {before - len(df):,}')

In [ ]:
# ── 4.5  Drop high-null columns and non-predictive columns ──high_null = df.columns[df.isnull().mean() > 0.5].tolist()print(f'High-null columns (>50% missing): {high_null}')useless = [    'CMPLNT_NUM',               # unique ID — not a feature    'CMPLNT_TO_DT', 'CMPLNT_TO_TM',    'Lat_Lon', 'New Georeferenced Column',    'RPT_DT',                   # report date ≈ complaint date    'STATION_NAME', 'PARKS_NM', 'HADEVELOPT',    'KY_CD',                    # numeric code for OFNS_DESC — redundant    'X_COORD_CD', 'Y_COORD_CD', # state plane coords — Lat/Lon is sufficient]drop_all = list(set(high_null + useless))df = df.drop(columns=[c for c in drop_all if c in df.columns])print(f'Total dropped: {len(drop_all)} | Remaining columns: {df.shape[1]}')

In [ ]:
# ── 4.6  Drop rows missing critical fields ──before = len(df)df = df.dropna(subset=['BORO_NM', 'OFNS_DESC', 'CMPLNT_FR_DT', 'LAW_CAT_CD'])print(f'Rows dropped (missing key fields): {before - len(df):,}')

In [ ]:
# ── 4.7  Standardise placeholder strings → NaN ──cat_cols = df.select_dtypes('object').columnsdf[cat_cols] = df[cat_cols].replace({'(null)': np.nan, 'UNKNOWN': np.nan, 'U': np.nan})print('Placeholder strings replaced.')

In [ ]:
# ── 4.8  Engineer time-based features ──df['HOUR']        = pd.to_datetime(df['CMPLNT_FR_TM'].astype(str), format='%H:%M:%S', errors='coerce').dt.hourdf['MONTH']       = df['CMPLNT_FR_DT'].dt.monthdf['DAY_OF_WEEK'] = df['CMPLNT_FR_DT'].dt.dayofweek   # 0=Mon 6=Sundf['IS_WEEKEND']  = (df['DAY_OF_WEEK'] >= 5).astype(int)df['YEAR']        = df['CMPLNT_FR_DT'].dt.year# TIME_BUCKET: 0=Night(0-5) 1=Morning(6-11) 2=Afternoon(12-17) 3=Evening(18-23)df['TIME_BUCKET'] = pd.cut(df['HOUR'], bins=[-1,5,11,17,23], labels=[0,1,2,3]).astype(float).astype('Int64')# Keep only 2020-2025 (removes stray pre-2020 historical entries)before = len(df)df = df[df['YEAR'].between(2020, 2025)]print(f'Rows outside 2020-2025 removed: {before - len(df):,}')print(f'\nFinal clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')print('\nYearly Complaint Counts:')print(df['YEAR'].value_counts().sort_index().to_string())

---## 5. Exploratory Data Analysis (EDA)

### 5.1  Year-over-Year Trends (2020–2025)

In [ ]:
# ── Annual totals + severity breakdown ──yearly = df['YEAR'].value_counts().sort_index()fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].bar(yearly.index.astype(str), yearly.values, color='steelblue', edgecolor='black')for i, (yr, cnt) in enumerate(zip(yearly.index, yearly.values)):    axes[0].text(i, cnt + 2000, f'{cnt:,}', ha='center', fontsize=9, fontweight='bold')axes[0].set_title('Total Complaints per Year (2020–2025)', fontweight='bold')axes[0].set_xlabel('Year')axes[0].set_ylabel('Complaints')sev_yr = df.groupby(['YEAR', 'LAW_CAT_CD']).size().unstack(fill_value=0)sev_yr.plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10', edgecolor='black')axes[1].set_title('Complaints by Severity per Year', fontweight='bold')axes[1].tick_params(axis='x', rotation=0)axes[1].legend(title='Severity')plt.tight_layout()plt.show()

In [ ]:
# ── Monthly seasonality — one line per year ──# Shows if seasonal patterns are consistent across yearsmonthly_yr = df.groupby(['YEAR','MONTH']).size().unstack(level=0)plt.figure(figsize=(12, 5))for yr in monthly_yr.columns:    plt.plot(monthly_yr.index, monthly_yr[yr], marker='o', markersize=4, label=str(yr))plt.title('Monthly Complaint Volume by Year (same shape = consistent seasonality)', fontweight='bold')plt.xlabel('Month')plt.ylabel('Complaints')plt.xticks(range(1,13))plt.legend(title='Year')plt.tight_layout()plt.show()

### 5.2  Temporal Patterns — When do crimes happen?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))monthly = df['MONTH'].value_counts().sort_index()axes[0].bar(monthly.index, monthly.values, color='steelblue')axes[0].set_title('Complaints by Month (all years)', fontweight='bold')axes[0].set_xlabel('Month')axes[0].set_xticks(range(1,13))day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']dow = df['DAY_OF_WEEK'].value_counts().sort_index()colors_dow = ['coral' if d >= 5 else '#6baed6' for d in dow.index]axes[1].bar(day_labels, dow.values, color=colors_dow)axes[1].set_title('Complaints by Day of Week (red=weekend)', fontweight='bold')hourly = df['HOUR'].value_counts().sort_index()axes[2].fill_between(hourly.index, hourly.values, alpha=0.3, color='seagreen')axes[2].plot(hourly.index, hourly.values, marker='o', color='seagreen', markersize=4)axes[2].set_title('Complaints by Hour of Day', fontweight='bold')axes[2].set_xticks(range(0,24))plt.tight_layout()plt.show()

In [ ]:
# ── Full 6-year daily trend with 7-day rolling average ──daily_vol = df.groupby(df['CMPLNT_FR_DT'].dt.date).size()plt.figure(figsize=(16, 4))plt.plot(daily_vol.index, daily_vol.values, linewidth=0.5, color='steelblue', alpha=0.6)plt.plot(daily_vol.rolling(7).mean().index, daily_vol.rolling(7).mean().values,         color='red', linewidth=1.5, label='7-day avg')plt.title('Daily Complaint Volume 2020–2025 (7-day rolling avg in red)', fontweight='bold')plt.xlabel('Date')plt.ylabel('Complaints / Day')plt.legend()plt.tight_layout()plt.show()

### 5.3  Categorical Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))top15 = df['OFNS_DESC'].value_counts().dropna().nlargest(15)sns.barplot(x=top15.values, y=top15.index, ax=axes[0,0], palette='Blues_r')axes[0,0].set_title('Top 15 Offense Types (all years)', fontweight='bold')axes[0,0].set_xlabel('Count')boro = df['BORO_NM'].value_counts().dropna()sns.barplot(x=boro.index, y=boro.values, ax=axes[0,1], palette='Set2')axes[0,1].set_title('Complaints by Borough (all years)', fontweight='bold')axes[0,1].tick_params(axis='x', rotation=20)severity = df['LAW_CAT_CD'].value_counts().dropna()axes[1,0].pie(severity.values, labels=severity.index,              autopct='%1.1f%%', startangle=140,              colors=['#d62728','#ff7f0e','#2ca02c'][:len(severity)])axes[1,0].set_title('Crime Severity Breakdown (Target #1)', fontweight='bold')outcome = df['CRM_ATPT_CPTD_CD'].value_counts().dropna()axes[1,1].bar(outcome.index, outcome.values, color=['#1f77b4','#ff7f0e'])axes[1,1].set_title('Completed vs Attempted (Target #2)', fontweight='bold')axes[1,1].set_yscale('log')for i,(lbl,val) in enumerate(zip(outcome.index,outcome.values)):    axes[1,1].text(i, val*1.1, f'{val:,}', ha='center', fontweight='bold')plt.tight_layout()plt.show()

### 5.4  Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))# Severity by boroughct = pd.crosstab(df['BORO_NM'], df['LAW_CAT_CD']).dropna()ct.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10')axes[0].set_title('Crime Severity by Borough', fontweight='bold')axes[0].tick_params(axis='x', rotation=30)axes[0].legend(title='Severity')# Severity mix (%) by year — has the felony rate changed?sev_yr_pct = df.groupby(['YEAR','LAW_CAT_CD']).size().unstack(fill_value=0)sev_yr_pct = sev_yr_pct.div(sev_yr_pct.sum(axis=1), axis=0) * 100sev_yr_pct.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')axes[1].set_title('Severity Mix (%) by Year', fontweight='bold')axes[1].tick_params(axis='x', rotation=0)axes[1].legend(title='Severity')plt.tight_layout()plt.show()

In [ ]:
# ── Heatmap: Top 10 Offense Types × Hour of Day ──top10_ofns = df['OFNS_DESC'].value_counts().dropna().nlargest(10).indexpt = pd.crosstab(df[df['OFNS_DESC'].isin(top10_ofns)]['OFNS_DESC'],                 df[df['OFNS_DESC'].isin(top10_ofns)]['HOUR'])plt.figure(figsize=(16, 7))sns.heatmap(pt, cmap='YlOrRd', linewidths=0.3, cbar_kws={'label': 'Count'})plt.title('Top 10 Offense Types × Hour of Day (all years)', fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# ── Geographic scatter: 30k sample coloured by severity ──sample_geo = df[['Latitude','Longitude','LAW_CAT_CD']].dropna().sample(min(30000,len(df)), random_state=42)plt.figure(figsize=(10, 9))cmap = {'FELONY':'#d62728','MISDEMEANOR':'#ff7f0e','VIOLATION':'#2ca02c'}for cat in ['VIOLATION','MISDEMEANOR','FELONY']:    g = sample_geo[sample_geo['LAW_CAT_CD']==cat]    if len(g): plt.scatter(g['Longitude'], g['Latitude'], s=0.3, alpha=0.3, label=cat, color=cmap.get(cat,'gray'))plt.title('NYC Crime Locations (30k sample) by Severity', fontweight='bold')plt.xlabel('Longitude'); plt.ylabel('Latitude')plt.legend(markerscale=12)plt.tight_layout()plt.show()

---## 6. Feature Engineering & ML Helper Functions

In [ ]:
# ── Base feature set ──# OFNS_DESC and PD_DESC are included here but EXCLUDED for Target 1 (leakage prevention)# They are valid features for Targets 2-4.BASE_FEATURES = [    'BORO_NM',            # Borough (5 categories)    'ADDR_PCT_CD',        # Precinct code (~77 unique)    'OFNS_DESC',          # Offense description  ← leaks LAW_CAT_CD, excluded for T1    'PD_DESC',            # Sub-offense description  ← same issue    'PREM_TYP_DESC',      # Premise type (street, residence, etc.)    'LOC_OF_OCCUR_DESC',  # Inside/outside/front/rear    'PATROL_BORO',        # Patrol borough    'JURISDICTION_CODE',  # Which agency (NYPD/transit/housing)    'SUSP_AGE_GROUP',     # Suspect age group    'SUSP_RACE',          # Suspect race    'SUSP_SEX',           # Suspect sex    'VIC_AGE_GROUP',      # Victim age group    'VIC_RACE',           # Victim race    'VIC_SEX',            # Victim sex    'HOUR',               # Hour of day (0-23)    'MONTH',              # Month (1-12)    'DAY_OF_WEEK',        # Day of week (0=Mon)    'IS_WEEKEND',         # Weekend binary flag    'TIME_BUCKET',        # Night/Morning/Afternoon/Evening    'YEAR',               # Year — captures multi-year trends    'Latitude',           # GPS latitude    'Longitude',          # GPS longitude]BASE_FEATURES = [f for f in BASE_FEATURES if f in df.columns]print(f'{len(BASE_FEATURES)} base features available.')

In [ ]:
def prepare_classification_data(target_col, features, dataframe, top_n_target=None):    """    Prepare X, y for classification.    - Drops rows with NaN in features or target    - Optionally filters to top_n most frequent classes    - Label-encodes all object columns    Returns: X, y, encoders_dict, feature_names    """    feats = [f for f in features if f != target_col]    data = dataframe[feats + [target_col]].dropna().copy()    if top_n_target is not None:        top_classes = data[target_col].value_counts().nlargest(top_n_target).index        data = data[data[target_col].isin(top_classes)]        print(f'  Filtered to top {top_n_target} classes: {list(top_classes)}')    le_dict = {}    for col in data.select_dtypes('object').columns:        le = LabelEncoder()        data[col] = le.fit_transform(data[col].astype(str))        le_dict[col] = le    X = data[feats].values    y = data[target_col].values    print(f'  Samples: {len(data):,}  |  Classes: {len(np.unique(y))}  |  Features: {len(feats)}')    return X, y, le_dict, featsdef train_and_evaluate(X_train, y_train, X_test, y_test, model, model_name, class_names=None):    """Train, predict, print full classification report, return metrics dict."""    model.fit(X_train, y_train)    y_pred = model.predict(X_test)    acc  = accuracy_score(y_test, y_pred)    f1   = f1_score(y_test, y_pred, average='weighted')    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)    print(f'\n{"="*55}')    print(f'  {model_name}')    print(f'  Accuracy: {acc:.4f}  F1: {f1:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}')    print(f'{"="*55}')    print(classification_report(y_test, y_pred, target_names=class_names))    return {'model_name': model_name, 'accuracy': acc, 'f1_weighted': f1,            'precision': prec, 'recall': rec, 'y_pred': y_pred, 'model_obj': model}def plot_comparison(res_lgb, res_cat, y_test, class_names, target_label):    """3-panel: metric bars + 2 confusion matrices."""    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))    metrics = pd.DataFrame([        {'Model':'LightGBM','Accuracy':res_lgb['accuracy'],'F1':res_lgb['f1_weighted']},        {'Model':'CatBoost','Accuracy':res_cat['accuracy'],'F1':res_cat['f1_weighted']},    ]).set_index('Model')    metrics.plot(kind='bar', ax=axes[0], color=['steelblue','coral'], edgecolor='black')    axes[0].set_title(f'{target_label}: Accuracy & F1', fontweight='bold')    axes[0].set_ylim(0, 1.1)    axes[0].tick_params(axis='x', rotation=0)    for c in axes[0].containers: axes[0].bar_label(c, fmt='%.3f', fontsize=8)    for i,(res,name) in enumerate([(res_lgb,'LightGBM'),(res_cat,'CatBoost')]):        ConfusionMatrixDisplay(confusion_matrix(y_test, res['y_pred']),                               display_labels=class_names).plot(ax=axes[i+1], colorbar=False, cmap='Blues')        axes[i+1].set_title(f'{name} Confusion Matrix', fontweight='bold')    plt.suptitle(f'Target: {target_label}', fontsize=13, fontweight='bold', y=1.02)    plt.tight_layout()    plt.show()def plot_feature_importance(model, feature_names, title):    """Horizontal bar chart of feature importances."""    imp = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=True)    plt.figure(figsize=(8, max(4, len(feature_names)*0.32)))    imp.plot(kind='barh', color='teal', edgecolor='black', linewidth=0.3)    plt.title(title, fontweight='bold')    plt.xlabel('Importance')    plt.tight_layout()    plt.show()print('Helper functions ready.')

---## 7. Target 1 — Crime Severity (`LAW_CAT_CD`)**Task:** 3-class — FELONY / MISDEMEANOR / VIOLATION**v1 problem:** 99.97% accuracy was data leakage — `OFNS_DESC` and `PD_DESC` are definitionally linked to severity in the NYPD system (e.g. 'FELONY ASSAULT' → FELONY). Including them lets the model look up the answer.**v2 fix:** These two columns are excluded. The model now uses genuine contextual signals: location, time of day, year, jurisdiction, demographics.

In [ ]:
# Exclude leaky features for this target onlysev_features = [f for f in BASE_FEATURES if f not in ('OFNS_DESC', 'PD_DESC')]print('Target 1: LAW_CAT_CD (Crime Severity)')print(f'Features (leaky cols excluded): {sev_features}')X1, y1, le1, feats1 = prepare_classification_data('LAW_CAT_CD', sev_features, df)X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y1, test_size=0.2, random_state=42, stratify=y1)class_names1 = le1['LAW_CAT_CD'].classes_ if 'LAW_CAT_CD' in le1 else Noneprint(f'Train: {X1_tr.shape}  |  Test: {X1_te.shape}')

In [ ]:
# ── LightGBM — class_weight='balanced' handles mild class imbalance ──lgb1 = lgb.LGBMClassifier(    n_estimators=500, max_depth=8, learning_rate=0.05,    num_leaves=63, min_child_samples=50,    subsample=0.8, colsample_bytree=0.8,    class_weight='balanced',    random_state=42, n_jobs=-1, verbose=-1)res1_lgb = train_and_evaluate(X1_tr, y1_tr, X1_te, y1_te, lgb1, 'LightGBM — LAW_CAT_CD', class_names1)

In [ ]:
# ── CatBoost — auto_class_weights='Balanced' is the CatBoost equivalent ──cat1 = CatBoostClassifier(    iterations=500, depth=8, learning_rate=0.05,    l2_leaf_reg=5, auto_class_weights='Balanced',    random_seed=42, verbose=0)res1_cat = train_and_evaluate(X1_tr, y1_tr, X1_te, y1_te, cat1, 'CatBoost — LAW_CAT_CD', class_names1)

In [ ]:
plot_comparison(res1_lgb, res1_cat, y1_te, class_names1, 'LAW_CAT_CD (leakage fixed in v2)')plot_feature_importance(res1_lgb['model_obj'], feats1, 'LightGBM Feature Importance — LAW_CAT_CD')

---## 8. Target 2 — Crime Outcome (`CRM_ATPT_CPTD_CD`)**Task:** Binary — COMPLETED vs ATTEMPTED**v1 problem:** ATTEMPTED class had F1=0.02 (LightGBM) and F1=0.11 (CatBoost). The class ratio was ~68:1 (COMPLETED:ATTEMPTED). Without balancing, the model just predicts COMPLETED every time.**v2 fix:** `class_weight='balanced'` forces the model to penalise errors on the rare ATTEMPTED class proportionally more.

In [ ]:
print('Target 2: CRM_ATPT_CPTD_CD (Completed vs Attempted)')X2, y2, le2, feats2 = prepare_classification_data('CRM_ATPT_CPTD_CD', BASE_FEATURES, df)X2_tr, X2_te, y2_tr, y2_te = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)class_names2 = le2['CRM_ATPT_CPTD_CD'].classes_ if 'CRM_ATPT_CPTD_CD' in le2 else None# Show class imbalance explicitlyvals, counts = np.unique(y2, return_counts=True)print('Class distribution:')for v,c in zip(vals,counts):    lbl = class_names2[v] if class_names2 is not None else v    print(f'  {lbl}: {c:,} ({c/len(y2)*100:.2f}%)')print(f'Train: {X2_tr.shape}  |  Test: {X2_te.shape}')

In [ ]:
lgb2 = lgb.LGBMClassifier(    n_estimators=500, max_depth=8, learning_rate=0.05,    num_leaves=63, min_child_samples=20,    subsample=0.8, colsample_bytree=0.8,    class_weight='balanced',  # upweights minority ATTEMPTED class    random_state=42, n_jobs=-1, verbose=-1)res2_lgb = train_and_evaluate(X2_tr, y2_tr, X2_te, y2_te, lgb2, 'LightGBM — CRM_ATPT_CPTD_CD', class_names2)

In [ ]:
cat2 = CatBoostClassifier(    iterations=500, depth=8, learning_rate=0.05,    l2_leaf_reg=5, auto_class_weights='Balanced',    random_seed=42, verbose=0)res2_cat = train_and_evaluate(X2_tr, y2_tr, X2_te, y2_te, cat2, 'CatBoost — CRM_ATPT_CPTD_CD', class_names2)

In [ ]:
plot_comparison(res2_lgb, res2_cat, y2_te, class_names2, 'CRM_ATPT_CPTD_CD (imbalance fixed in v2)')plot_feature_importance(res2_lgb['model_obj'], feats2, 'LightGBM Feature Importance — CRM_ATPT_CPTD_CD')

---## 9. Target 3 — Borough Prediction (`BORO_NM`)**Task:** 5-class — MANHATTAN / BROOKLYN / BRONX / QUEENS / STATEN ISLANDLatitude, Longitude, PATROL_BORO, and ADDR_PCT_CD are excluded (they directly encode the borough).The model must infer location from crime type, time, jurisdiction, and demographics.

In [ ]:
boro_features = [f for f in BASE_FEATURES                 if f not in ('BORO_NM','Latitude','Longitude','PATROL_BORO','ADDR_PCT_CD')]print('Target 3: BORO_NM (Borough prediction — geo excluded)')X3, y3, le3, feats3 = prepare_classification_data('BORO_NM', boro_features, df)X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.2, random_state=42, stratify=y3)class_names3 = le3['BORO_NM'].classes_ if 'BORO_NM' in le3 else Noneprint(f'Train: {X3_tr.shape}  |  Test: {X3_te.shape}')

In [ ]:
lgb3 = lgb.LGBMClassifier(    n_estimators=500, max_depth=8, learning_rate=0.05,    num_leaves=63, min_child_samples=50,    subsample=0.8, colsample_bytree=0.8,    random_state=42, n_jobs=-1, verbose=-1)res3_lgb = train_and_evaluate(X3_tr, y3_tr, X3_te, y3_te, lgb3, 'LightGBM — BORO_NM', class_names3)

In [ ]:
cat3 = CatBoostClassifier(    iterations=500, depth=8, learning_rate=0.05,    l2_leaf_reg=5, random_seed=42, verbose=0)res3_cat = train_and_evaluate(X3_tr, y3_tr, X3_te, y3_te, cat3, 'CatBoost — BORO_NM', class_names3)

In [ ]:
plot_comparison(res3_lgb, res3_cat, y3_te, class_names3, 'BORO_NM (Borough)')plot_feature_importance(res3_lgb['model_obj'], feats3, 'LightGBM Feature Importance — BORO_NM')

---## 10. Target 4 — Offense Type (`OFNS_DESC`, top 10)**Task:** 10-class — what type of crime?`PD_DESC` excluded (it's a detailed sub-code that directly identifies the offense).

In [ ]:
ofns_features = [f for f in BASE_FEATURES if f not in ('OFNS_DESC','PD_DESC')]print('Target 4: OFNS_DESC (Offense Type — top 10 classes)')X4, y4, le4, feats4 = prepare_classification_data('OFNS_DESC', ofns_features, df, top_n_target=10)X4_tr, X4_te, y4_tr, y4_te = train_test_split(X4, y4, test_size=0.2, random_state=42, stratify=y4)class_names4 = le4['OFNS_DESC'].classes_ if 'OFNS_DESC' in le4 else Noneprint(f'Train: {X4_tr.shape}  |  Test: {X4_te.shape}')

In [ ]:
lgb4 = lgb.LGBMClassifier(    n_estimators=500, max_depth=8, learning_rate=0.05,    num_leaves=63, min_child_samples=50,    subsample=0.8, colsample_bytree=0.8,    class_weight='balanced',    random_state=42, n_jobs=-1, verbose=-1)res4_lgb = train_and_evaluate(X4_tr, y4_tr, X4_te, y4_te, lgb4, 'LightGBM — OFNS_DESC', class_names4)

In [ ]:
cat4 = CatBoostClassifier(    iterations=500, depth=8, learning_rate=0.05,    l2_leaf_reg=5, auto_class_weights='Balanced',    random_seed=42, verbose=0)res4_cat = train_and_evaluate(X4_tr, y4_tr, X4_te, y4_te, cat4, 'CatBoost — OFNS_DESC', class_names4)

In [ ]:
plot_comparison(res4_lgb, res4_cat, y4_te, class_names4, 'OFNS_DESC (Offense Type — Top 10)')plot_feature_importance(res4_lgb['model_obj'], feats4, 'LightGBM Feature Importance — OFNS_DESC')

---## 11. Target 5 — Daily Case Count (Regression)**Task:** Predict total complaints per day**v1 R²: −3.54** — Catastrophic. Root cause: only ~365 daily data points. LAG_365 was all NaN (no year-ago data). Models were worse than just predicting the mean.**v2:** ~2,200 daily data points (2020–2025). LAG_365 is fully populated for all test rows. Expected R² > 0.70.

In [ ]:
# ── Build daily aggregate dataset ──df_daily = df.assign(DATE=df['CMPLNT_FR_DT'].dt.date).dropna(subset=['DATE'])daily_counts = df_daily.groupby('DATE').size().reset_index(name='DAILY_CASE_COUNT')daily_counts['DATE'] = pd.to_datetime(daily_counts['DATE'])daily_counts = daily_counts.sort_values('DATE').reset_index(drop=True)# ── Temporal features ──daily_counts['MONTH']        = daily_counts['DATE'].dt.monthdaily_counts['DAY_OF_WEEK']  = daily_counts['DATE'].dt.dayofweekdaily_counts['IS_WEEKEND']   = (daily_counts['DAY_OF_WEEK'] >= 5).astype(int)daily_counts['DAY_OF_MONTH'] = daily_counts['DATE'].dt.daydaily_counts['WEEK_OF_YEAR'] = daily_counts['DATE'].dt.isocalendar().week.astype(int)daily_counts['YEAR']         = daily_counts['DATE'].dt.year# ── Lag features — short-term memory ──daily_counts['LAG_1']   = daily_counts['DAILY_CASE_COUNT'].shift(1)    # yesterdaydaily_counts['LAG_7']   = daily_counts['DAILY_CASE_COUNT'].shift(7)    # same day last weekdaily_counts['LAG_14']  = daily_counts['DAILY_CASE_COUNT'].shift(14)daily_counts['LAG_30']  = daily_counts['DAILY_CASE_COUNT'].shift(30)daily_counts['LAG_365'] = daily_counts['DAILY_CASE_COUNT'].shift(365)  # same day last year# ── Rolling averages — trend features ──daily_counts['ROLLING_7']  = daily_counts['DAILY_CASE_COUNT'].rolling(7).mean()daily_counts['ROLLING_30'] = daily_counts['DAILY_CASE_COUNT'].rolling(30).mean()daily_counts['ROLLING_90'] = daily_counts['DAILY_CASE_COUNT'].rolling(90).mean()# Drop NaN rows (first 365 days before LAG_365 is populated)daily_counts = daily_counts.dropna()print(f'Daily dataset: {daily_counts.shape[0]} rows')print(f'Date range: {daily_counts["DATE"].min().date()} → {daily_counts["DATE"].max().date()}')print(f'Target: {daily_counts["DAILY_CASE_COUNT"].min():.0f} – {daily_counts["DAILY_CASE_COUNT"].max():.0f} complaints/day')

In [ ]:
REG_FEATURES = [    'MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND', 'DAY_OF_MONTH', 'WEEK_OF_YEAR', 'YEAR',    'LAG_1', 'LAG_7', 'LAG_14', 'LAG_30', 'LAG_365',    'ROLLING_7', 'ROLLING_30', 'ROLLING_90']X5 = daily_counts[REG_FEATURES].valuesy5 = daily_counts['DAILY_CASE_COUNT'].values# Chronological 80/20 split — NEVER shuffle time-series datasplit_idx = int(len(X5) * 0.8)X5_tr, X5_te = X5[:split_idx], X5[split_idx:]y5_tr, y5_te = y5[:split_idx], y5[split_idx:]dates_test = daily_counts['DATE'].values[split_idx:]print(f'Train: {X5_tr.shape}  |  Test: {X5_te.shape}')

In [ ]:
lgb5 = lgb.LGBMRegressor(    n_estimators=1000, max_depth=6, learning_rate=0.03,    num_leaves=31, min_child_samples=10,    subsample=0.8, colsample_bytree=0.8,    random_state=42, n_jobs=-1, verbose=-1)lgb5.fit(X5_tr, y5_tr)y5_pred_lgb = lgb5.predict(X5_te)mae_lgb  = mean_absolute_error(y5_te, y5_pred_lgb)rmse_lgb = np.sqrt(mean_squared_error(y5_te, y5_pred_lgb))r2_lgb   = r2_score(y5_te, y5_pred_lgb)print(f'LightGBM Regression:')print(f'  MAE  : {mae_lgb:.1f} complaints/day')print(f'  RMSE : {rmse_lgb:.1f}')print(f'  R²   : {r2_lgb:.4f}   (v1 was -3.54)')

In [ ]:
cat5 = CatBoostRegressor(    iterations=1000, depth=6, learning_rate=0.03,    l2_leaf_reg=5, random_seed=42, verbose=0)cat5.fit(X5_tr, y5_tr)y5_pred_cat = cat5.predict(X5_te)mae_cat  = mean_absolute_error(y5_te, y5_pred_cat)rmse_cat = np.sqrt(mean_squared_error(y5_te, y5_pred_cat))r2_cat   = r2_score(y5_te, y5_pred_cat)print(f'CatBoost Regression:')print(f'  MAE  : {mae_cat:.1f} complaints/day')print(f'  RMSE : {rmse_cat:.1f}')print(f'  R²   : {r2_cat:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))axes[0,0].plot(dates_test, y5_te, color='black', linewidth=1, label='Actual')axes[0,0].plot(dates_test, y5_pred_lgb, color='steelblue', linewidth=1, alpha=0.8, label='LightGBM')axes[0,0].set_title(f'LightGBM: Actual vs Predicted  (R²={r2_lgb:.3f})', fontweight='bold')axes[0,0].legend(); axes[0,0].set_ylabel('Complaints / Day')axes[0,1].plot(dates_test, y5_te, color='black', linewidth=1, label='Actual')axes[0,1].plot(dates_test, y5_pred_cat, color='coral', linewidth=1, alpha=0.8, label='CatBoost')axes[0,1].set_title(f'CatBoost: Actual vs Predicted  (R²={r2_cat:.3f})', fontweight='bold')axes[0,1].legend()mn, mx = y5_te.min()*0.95, y5_te.max()*1.05axes[1,0].scatter(y5_te, y5_pred_lgb, s=10, alpha=0.5, color='steelblue', label='LightGBM')axes[1,0].scatter(y5_te, y5_pred_cat, s=10, alpha=0.5, color='coral', label='CatBoost')axes[1,0].plot([mn,mx],[mn,mx],'k--',linewidth=1,label='Perfect')axes[1,0].set_title('Predicted vs Actual Scatter', fontweight='bold')axes[1,0].set_xlabel('Actual'); axes[1,0].set_ylabel('Predicted')axes[1,0].legend()pd.DataFrame({'Metric':['MAE','RMSE','R²'],              'LightGBM':[mae_lgb,rmse_lgb,r2_lgb],              'CatBoost':[mae_cat,rmse_cat,r2_cat]}).set_index('Metric').plot(    kind='bar', ax=axes[1,1], color=['steelblue','coral'], edgecolor='black')axes[1,1].set_title('Regression Metrics Comparison', fontweight='bold')axes[1,1].tick_params(axis='x', rotation=0)for c in axes[1,1].containers: axes[1,1].bar_label(c, fmt='%.1f', fontsize=8)plt.tight_layout()plt.show()

In [ ]:
plot_feature_importance(lgb5, REG_FEATURES, 'LightGBM Feature Importance — Daily Case Count')

---## 12. Grand Comparison — All 5 Targets × 2 Models

In [ ]:
grand_results = pd.DataFrame([    {'Target':'LAW_CAT_CD (Severity)',      'Model':'LightGBM','Task':'Classification',     'Accuracy':res1_lgb['accuracy'],'F1':res1_lgb['f1_weighted'],'Precision':res1_lgb['precision'],'Recall':res1_lgb['recall']},    {'Target':'LAW_CAT_CD (Severity)',      'Model':'CatBoost','Task':'Classification',     'Accuracy':res1_cat['accuracy'],'F1':res1_cat['f1_weighted'],'Precision':res1_cat['precision'],'Recall':res1_cat['recall']},    {'Target':'CRM_ATPT_CPTD_CD (Outcome)','Model':'LightGBM','Task':'Classification',     'Accuracy':res2_lgb['accuracy'],'F1':res2_lgb['f1_weighted'],'Precision':res2_lgb['precision'],'Recall':res2_lgb['recall']},    {'Target':'CRM_ATPT_CPTD_CD (Outcome)','Model':'CatBoost','Task':'Classification',     'Accuracy':res2_cat['accuracy'],'F1':res2_cat['f1_weighted'],'Precision':res2_cat['precision'],'Recall':res2_cat['recall']},    {'Target':'BORO_NM (Borough)',          'Model':'LightGBM','Task':'Classification',     'Accuracy':res3_lgb['accuracy'],'F1':res3_lgb['f1_weighted'],'Precision':res3_lgb['precision'],'Recall':res3_lgb['recall']},    {'Target':'BORO_NM (Borough)',          'Model':'CatBoost','Task':'Classification',     'Accuracy':res3_cat['accuracy'],'F1':res3_cat['f1_weighted'],'Precision':res3_cat['precision'],'Recall':res3_cat['recall']},    {'Target':'OFNS_DESC (Offense Type)',   'Model':'LightGBM','Task':'Classification',     'Accuracy':res4_lgb['accuracy'],'F1':res4_lgb['f1_weighted'],'Precision':res4_lgb['precision'],'Recall':res4_lgb['recall']},    {'Target':'OFNS_DESC (Offense Type)',   'Model':'CatBoost','Task':'Classification',     'Accuracy':res4_cat['accuracy'],'F1':res4_cat['f1_weighted'],'Precision':res4_cat['precision'],'Recall':res4_cat['recall']},    {'Target':'Daily Case Count',           'Model':'LightGBM','Task':'Regression',     'Accuracy':r2_lgb,'F1':np.nan,'Precision':np.nan,'Recall':np.nan,'MAE':mae_lgb,'RMSE':rmse_lgb},    {'Target':'Daily Case Count',           'Model':'CatBoost','Task':'Regression',     'Accuracy':r2_cat,'F1':np.nan,'Precision':np.nan,'Recall':np.nan,'MAE':mae_cat,'RMSE':rmse_cat},])clf = grand_results[grand_results['Task']=='Classification'][['Target','Model','Accuracy','F1','Precision','Recall']].copy()for col in ['Accuracy','F1','Precision','Recall']: clf[col] = clf[col].map('{:.4f}'.format)print('='*85)print('  CLASSIFICATION RESULTS (v2 — leakage & imbalance fixed)')print('='*85)print(clf.to_string(index=False))reg = grand_results[grand_results['Task']=='Regression'][['Target','Model','MAE','RMSE','Accuracy']].copy().rename(columns={'Accuracy':'R²'})for col in ['MAE','RMSE','R²']: reg[col] = reg[col].map('{:.4f}'.format)print('\n--- Regression ---')print(reg.to_string(index=False))

In [ ]:
clf_df = grand_results[grand_results['Task']=='Classification'].copy()clf_df['label'] = clf_df['Model'] + '\n' + clf_df['Target'].str.split('(').str[0].str.strip()fig, ax = plt.subplots(figsize=(15, 6))x = np.arange(len(clf_df))w = 0.35b1 = ax.bar(x-w/2, clf_df['Accuracy'], width=w, label='Accuracy', color='steelblue', edgecolor='black', lw=0.3)b2 = ax.bar(x+w/2, clf_df['F1'],       width=w, label='F1 (weighted)', color='coral', edgecolor='black', lw=0.3)ax.set_xticks(x)ax.set_xticklabels(clf_df['label'], fontsize=8, rotation=30, ha='right')ax.set_ylim(0, 1.12)ax.set_ylabel('Score')ax.set_title('Grand Comparison — v2 (leakage & imbalance fixed)', fontweight='bold', fontsize=12)ax.legend()for bar in list(b1)+list(b2):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,            f'{bar.get_height():.3f}', ha='center', fontsize=7)plt.tight_layout()plt.show()

In [ ]:
print('='*70)print('  WINNER ANALYSIS')print('='*70)for target in grand_results['Target'].unique():    subset = grand_results[grand_results['Target']==target]    is_reg = subset['Task'].iloc[0]=='Regression'    best = subset.loc[subset['Accuracy'].idxmax()]    diff = abs(subset['Accuracy'].values[0] - subset['Accuracy'].values[1])    metric = 'R²' if is_reg else 'Acc'    f1_str = '' if is_reg else f'  F1={best["F1"]:.4f}'    mae_str = f'  MAE={best["MAE"]:.1f}' if is_reg else ''    print(f'\n  {target}')    print(f'    Winner : {best["Model"]}  |  {metric}={best["Accuracy"]:.4f}{f1_str}{mae_str}')    print(f'    Margin : {diff:.4f}')print('\n'+'='*70)

---## 13. Score Analysis — What the Numbers Mean### Target 1 — `LAW_CAT_CD` (Crime Severity)- **v1: 99.97%** — data leakage (OFNS_DESC encodes severity definitionally)- **v2:** Genuine accuracy; reflects what context alone can predict- Expected range: 55–75% depending on how much information location/time carry### Target 2 — `CRM_ATPT_CPTD_CD` (Outcome)- **v1 ATTEMPTED F1: 0.02** — model predicted COMPLETED every time- **v2:** `class_weight='balanced'` forces attention on ATTEMPTED- Overall accuracy may drop slightly; ATTEMPTED recall rises significantly- This is the more operationally useful result — identifying ATTEMPTED crimes matters### Target 3 — `BORO_NM` (Borough)- **v1: 43%** (random 5-class baseline = 20%)- **v2 improvement:** More training data (5 years) + YEAR captures borough-level shifts- Crime types are unevenly distributed by borough — offense type is a strong signal### Target 4 — `OFNS_DESC` (Offense Type)- **v1: 37.8%** on top-10 classes- **v2 improvement:** 5× more training data + balanced weights for rare offense types- Some classes are genuinely hard to distinguish without offense-defining features### Target 5 — Daily Case Count (Regression)- **v1 R²: −3.54** — only 365 rows; LAG_365 was all NaN- **v2:** ~2,200 rows; LAG_365 fully populated- Expected R² > 0.70 — the year-ago lag is the single strongest feature### About Model Choice- Both LightGBM and CatBoost are competitive; differences are usually < 1%- LightGBM wins on speed; CatBoost often wins on datasets with many categorical features- The gap between them narrows as dataset size grows